# Workshop 6. Simple linear regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture06/workshop06.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the regression from the lecture in code, fully worked: run it, answer the questions and change one thing.
Tasks 2 to 4 start from an empty cell: visits and age, visits and years as a customer, and the slope and its
standard error by hand for the customers in Nordjylland. The extension tasks are for those who have time left.
Exercises in the style of the textbook, with small numbers and no data, are in the TA's separate notebook for this
session (link on Moodle).

Each core task has two hints: the first names the statistical step, the second names the functions. Try the task
before you open them. The toolbox says where you have seen each function before, so you can look it up in that
notebook. The code patterns, the slides and the lecture notebook are your main references. You are welcome to use an
LLM, but every task can be solved without one, and you should be able to explain the statistics in your answers
yourself.

In every task, say what y and x are and in which units before you fit anything. Each answer is in words a manager can
follow, with the numbers it rests on. The tests are two-sided, as in the statsmodels output.

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `smf.ols("y ~ x", data=df).fit()` | Fits the least-squares line of y on x, with an intercept | Lecture 6, Part 1 |
| `model.summary()` | The full output table | Lecture 6, Part 3 |
| `model.params`, `model.bse` | b<sub>0</sub> and b<sub>1</sub>; their standard errors | Lecture 6, Parts 1 and 3 |
| `model.tvalues`, `model.pvalues`, `model.conf_int()` | t, the two-sided p-value for H<sub>0</sub>: β = 0, and the 95% intervals | Lecture 6, Part 3 |
| `model.rsquared`, `np.sqrt(model.scale)` | R² and s<sub>e</sub> | Lecture 6, Part 2 |
| `model.predict(pd.DataFrame({"x": [value]}))` | The line's value at a given x | Lecture 6, Part 4 |
| `df[df["col"] == value]` | The rows where a column has a given value | Lecture 6, Exercise 4; Lecture 4 (Copenhagen and Aarhus orders) |
| `df["col"].mean()`, `.std(ddof=1)`, `.min()`, `.max()` | Mean, standard deviation (dividing by n − 1), smallest and largest value | Lectures 1 and 2 |
| `plt.scatter(x, y)`, `plt.plot(xs, ys)` | Scatter plot; a line through given points | Lecture 6, Part 1 |
| `stats.t.ppf(0.975, df=n - 2)` | The two-sided critical value for α = 0.05 | Lecture 6, Part 3; Lecture 3 |
| `np.sqrt(x)` | Square root | Lecture 1 |

## Code patterns

Two complete patterns for today. Replace `df`, `y` and `x` with your data and column names. The patterns are
templates; they do not run as they stand.

**Fit a line, print the output, and pull out the numbers**

```python
model = smf.ols("y ~ x", data=df).fit()
print(model.summary())
b0, b1 = model.params["Intercept"], model.params["x"]
s_b1 = model.bse["x"]
lo, hi = model.conf_int().loc["x"]
print(f"b0 = {b0:.2f}, b1 = {b1:.3f}, s_b1 = {s_b1:.3f}, interval {lo:.2f} to {hi:.2f}")
print(f"R2 = {model.rsquared:.3f}, s_e = {np.sqrt(model.scale):.2f}")
```

**A scatter plot with the line, and a prediction**

```python
plt.scatter(df["x"], df["y"], s=8, alpha=0.5)
xs = np.array([df["x"].min(), df["x"].max()])
plt.plot(xs, b0 + b1 * xs, color="darkorange")
plt.xlabel("x and its unit"); plt.ylabel("y and its unit")
plt.show()
print(model.predict(pd.DataFrame({"x": [5]})))   # the line's value at x = 5
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 RetailCorp customers drawn at random
customers.head()

## Core tasks

### Task 1. Spend and years as a customer (worked)

The lecture walkthrough in code. Run the cells and answer the questions below them.

y = `spend_2025` (DKK in 2025), x = `tenure_years` (years as a customer on 1 January 2025).

In [ ]:
data = customers      # step 3: change to customers[customers["member"] == 1], then to customers[customers["member"] == 0]

model = smf.ols("spend_2025 ~ tenure_years", data=data).fit()
print(model.summary())

In [ ]:
b0, b1 = model.params["Intercept"], model.params["tenure_years"]
lo, hi = model.conf_int().loc["tenure_years"]
print(f"n = {int(model.nobs)}, b0 = {b0:.2f}, b1 = {b1:.3f}, s_b1 = {model.bse['tenure_years']:.3f}, "
      f"interval {lo:.2f} to {hi:.2f}")
print(f"R2 = {model.rsquared:.3f}, s_e = {np.sqrt(model.scale):.2f}")

plt.scatter(data["tenure_years"], data["spend_2025"], s=8, alpha=0.5)
xs = np.array([0, 12])
plt.plot(xs, b0 + b1 * xs, color="darkorange")
plt.xlabel("Years as a customer"); plt.ylabel("Spend in 2025, DKK")
plt.show()
print(model.predict(pd.DataFrame({"tenure_years": [5]})))

**Questions**

1. Find b<sub>1</sub>, s<sub>b1</sub>, t, the p-value, the 95% interval and R² in the output. Say in one sentence each
   what b<sub>1</sub> and R² mean for RetailCorp.
2. Where in the output is the test of H<sub>0</sub>: β<sub>1</sub> = 0, and what do you conclude at α = 0.05?
3. Why does the output show `0.000` for the p-value, and how do you report it?

**Change one thing.** In the first cell, change `data = customers` to club members only,
`customers[customers["member"] == 1]`, and run both cells. Then do the same for non-members (`== 0`). Compare the two
slopes and their intervals.

**Your answer:**

### Task 2. Do older customers visit less?

RetailCorp's web team asks whether older customers use the shop less often. Fit the line of the number of logged-in
visits in 2025 (`visits_2025`) on `age`, and print the output.

**Answer in words.**

1. Interpret b<sub>1</sub> with its unit. Test H<sub>0</sub>: β<sub>1</sub> = 0 at α = 0.05 (two-sided, as in the
   output) and give the 95% interval.
2. Predict the number of visits of a 25-year-old and of a 65-year-old customer. How far apart are the two age groups
   on average?
3. How much does age help to predict the visits of a single customer? Use R², and compare s<sub>e</sub> with the
   standard deviation of visits.

<details>
<summary>Hint 1</summary>

y is `visits_2025` and x is `age`, so the slope is in visits per year of age. Question 2 is about the average
difference between age groups (the line); question 3 is about how far single customers lie from the line.
</details>

<details>
<summary>Hint 2</summary>

`smf.ols("visits_2025 ~ age", data=customers).fit()`, `.summary()`, `.predict(pd.DataFrame({"age": [25, 65]}))`,
`np.sqrt(model.scale)`, `customers["visits_2025"].std(ddof=1)` (see the toolbox).
</details>

**Your answer:**

### Task 3. Do long-time customers visit more?

Is there a relationship between years as a customer (`tenure_years`) and the number of visits in 2025? Fit the line,
make a scatter plot with the line, and test H<sub>0</sub>: β<sub>1</sub> = 0 against a two-sided H<sub>1</sub> at
α = 0.05.

**Answer in words.** What can RetailCorp conclude? Does the result show that visits do not depend on tenure? Use the
interval in your answer. Then write the result in three or four sentences, as on the slide "Reporting a regression": what was regressed on what and n, the slope with its interval, the test result and R², and what the data
allow you to conclude.

<details>
<summary>Hint 1</summary>

The test and the interval are in the row for `tenure_years`. If you do not reject H<sub>0</sub>, the data are
compatible with a slope of 0, and the interval shows which other slopes are also compatible with the data.
</details>

<details>
<summary>Hint 2</summary>

`smf.ols("visits_2025 ~ tenure_years", data=customers).fit()`, `.summary()`, `.conf_int()`; the scatter plot is the
second code pattern.
</details>

**Your answer:**

### Task 4. The slope by hand for Nordjylland

At the oral exam you may be asked to compute or explain the formulas. For the 108 customers in Nordjylland, Python
gave these summary numbers for spend (y) and years as a customer (x):

| Quantity | Value |
|---|---|
| n | 108 |
| x̄ (years) | 3.2343 |
| ȳ (DKK) | 2,063.9757 |
| Σ(x − x̄)² | 845.5632 |
| Σ(x − x̄)(y − ȳ) | 227,879.0198 |
| SSE | 165,730,107.57 |

Compute b<sub>1</sub>, b<sub>0</sub>, s<sub>e</sub>, s<sub>b1</sub> and t with a calculator or with Python as a
calculator (type in the numbers). Find the critical value, decide whether to reject H<sub>0</sub>: β<sub>1</sub> = 0
at α = 0.05, and compute the 95% interval. Then fit the line on the Nordjylland customers and check your numbers
against the output.

**Answer in words.** Write one sentence for a manager about customers in Nordjylland.

<details>
<summary>Hint 1</summary>

b<sub>1</sub> = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)², b<sub>0</sub> = ȳ − b<sub>1</sub>x̄,
s<sub>e</sub> = √(SSE / (n − 2)), s<sub>b1</sub> = s<sub>e</sub> / √Σ(x − x̄)², t = b<sub>1</sub> / s<sub>b1</sub>,
interval b<sub>1</sub> ± t<sub>n−2, 0.025</sub> s<sub>b1</sub>.
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt(...)`, `stats.t.ppf(0.975, df=n - 2)`. For the check, pick the rows with
`customers[customers["region"] == "Nordjylland"]` (see the toolbox) and fit with `smf.ols`.
</details>

**Your answer:**

## Extension tasks

### Extension 1. Airline delays

In the airline passenger survey from Workshop 3, regress the arrival delay on the departure delay (both in minutes).
Interpret b<sub>1</sub> and b<sub>0</sub>, and explain why the slope is close to 1 and the intercept close to 0. How
many rows does the model use, and why is that fewer than the number of rows in the file? The file:
`pd.read_csv(url + "airline_survey.csv")`. Put a column name with spaces in `Q('...')` inside the formula.

**Your answer:**

### Extension 2. Two variables at once?

In the lecture, age and years as a customer each accounted for about 13% of the variation in spend. Does that mean
that together they account for about 26%? Compute the correlation between age and tenure, and write down your guess
and a reason. (Lecture 7 gives the answer.)

**Your answer:**

### Extension 3. The exam assignment

If your group has time left, start on sections 1 to 4 of the exam assignment (link on Moodle).